In [1]:
import sys
import os
import torch
import numpy as np
from torch.distributions import Categorical
import matplotlib.pyplot as plt
from environment2 import TradingEnv
from models import GRUActorCritic
from utils import load_and_preprocess_data
import logging
from tensorboardX import SummaryWriter
import net_utils

import numpy as np
import matplotlib.pyplot as plt

In [2]:
logger = None

np.set_printoptions(suppress=True)

total_updates = 100000
rollout_length = 1024
gamma = 0.99
clip_epsilon = 0.2
ppo_epochs = 4
lr = 3e-4
window_size = 64
hidden_dim = 256

In [1]:
# os.environ['OPENAI_API_KEY']

In [4]:
file_path = os.path.expanduser('~/trader/data/BTC-2021min.csv')
df = load_and_preprocess_data(file_path)

env = TradingEnv(df, logger=logger, window_size=window_size)
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

close ok
low ok
high ok
volume ok


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
model = GRUActorCritic(256, hidden_dim, n_actions=n_actions).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

Device: cuda


In [6]:
obs_dim, n_actions

(64, 3)

In [7]:
# Initialize the writer (set a log directory as needed)
writer = SummaryWriter(log_dir='./logs')

all_rewards, all_actions = [], []
global_step = 0

In [8]:
states, actions, rewards, masks, log_probs, values = [], [], [], [], [], []
ep_reward = 0
hidden_state = None
state = env.reset()

close ok
low ok
high ok
volume ok


In [9]:
input_tensor = torch.FloatTensor(state).to(device).unsqueeze(0)
input_tensor.shape

torch.Size([1, 256])

In [10]:
logits, value, hidden_state = model(input_tensor, hidden_state)

In [11]:
logits.shape

torch.Size([1, 3])

In [ ]:
for step in range(rollout_length):

    input_tensor = torch.FloatTensor(state).to(device).unsqueeze(0)
    logits, value, hidden_state = model(input_tensor, hidden_state)
    dist = Categorical(logits=logits.squeeze(1))
    action = dist.sample()

    log_prob = dist.log_prob(action)

    next_state, reward, done, info = env.step(action.item())
    ep_reward += reward

    # Save rollout data
    states.append(state)
    actions.append(action.item())
    all_actions.append(action.item())
    rewards.append(reward)
    masks.append(1 - float(done))
    log_probs.append(log_prob.item())
    values.append(value.item())

    state = next_state
    global_step += 1

    if done:
        state = env.reset()
        all_rewards.append(ep_reward)
        ep_reward = 0
        hidden_state = None

    if env.total_balance < 1.0:
        state = env.reset()

In [ ]:
ep_reward

In [ ]:
states = np.array(states)
states.shape

In [ ]:
states = torch.FloatTensor(states).to(device) # .transpose(1, 2)
actions = torch.LongTensor(actions).to(device)
log_probs_old = torch.FloatTensor(log_probs).to(device)
values = torch.FloatTensor(values).to(device)
rewards = torch.FloatTensor(rewards).to(device)
masks = torch.FloatTensor(masks).to(device)

In [ ]:
state_tensor = torch.FloatTensor(state).to(device).unsqueeze(0)
state_tensor

In [ ]:
logits0, next_value, hidden_state0 = model(input_tensor, hidden_state)

In [ ]:
next_value

In [ ]:
values.shape

In [ ]:
hidden_state

In [ ]:
values = torch.cat((values, torch.FloatTensor([next_value]).to(device)))
values.shape

In [ ]:
advantages = net_utils.compute_gae(rewards.cpu().numpy(),
                                   masks.cpu().numpy(),
                                   values.cpu().numpy(),
                                   gamma,
                                   lam=0.95)
len(advantages)

In [ ]:
advantages = torch.FloatTensor(advantages).to(device)
advantages.shape


In [ ]:
returns = net_utils.compute_returns(rewards.cpu().numpy(), masks.cpu().numpy(), values.cpu().numpy(), gamma)
returns = torch.FloatTensor(returns).to(device)

In [ ]:
returns[:5]

In [ ]:
mean_loss = net_utils.ppo_update(model, optimizer, states, actions, log_probs_old,
                                 returns, advantages, clip_epsilon, ppo_epochs)
mean_loss

In [ ]:
states = torch.FloatTensor(states).to(device)
states.shape

In [ ]:
actions = torch.LongTensor(actions).to(device)
actions

In [ ]:
masks = torch.FloatTensor(masks).to(device)
masks

In [ ]:
log_probs_old = torch.FloatTensor(log_probs).to(device)
log_probs_old

In [ ]:
values = torch.FloatTensor(values).to(device)
values

In [ ]:
values.min(), values.max()

In [ ]:
rewards = torch.FloatTensor(rewards).to(device)
rewards